In [1]:
import numpy as np
import pandas as pd
import os
from faker import Faker

In [8]:
def generate_loan_data(n=227000,seed = 42):
    """ Generate synthetic loan dataset """
    fake=Faker('en_IN')
    np.random.seed(seed)

    print(f'Generating {n} synthetic loan dataset.....')

    data = {
        'application_id': [f'LN{fake.unique.random_number(digits=8)}' for _ in range(n)],
        'age' : np.random.randint(21,65,n),
        'gender' : np.random.choice(['M','F'],n,p=[0.65,0.35]),
        'marital_status': np.random.choice(['single','married','divorced'],n,p=[0.3,0.6,0.1]),
        'dependents':np.random.choice([0,1,2,3,4],n,p=[0.25,0.3,0.25,0.15,0.05]),
        'education':np.random.choice(['graduate','post_graduate','under_graduate'],n,p=[0.45,0.25,0.30]),
        'employment_type':np.random.choice(['salaried','self_employed','business'],n,p=[0.55,0.25,0.20]),
        'annual_income_lakhs': np.round(np.random.lognormal(2.5,0.7,n).clip(1.5,100),2),
        'loan_amount_lakhs': np.round(np.random.lognormal(2.0,0.8,n).clip(0.5,200),2),
        'loan_tenure_months':np.random.choice([12,24,36,48,60,84,120,180,240],n),
        'interest_rate':np.round(np.random.uniform(7.5,16.5,n),2),
        'credit_score' : np.random.randint(300,900,n),
        'existing_loans':np.random.choice([0,1,2,3,4],n,p=[0.35,0.30,0.20,0.10,0.05]),
        'credit_history_years' : np.random.randint(0,25,n),
        'monthly_emi_existing' : np.round(np.random.lognormal(2.0,1.0,n).clip(0,80000),0),
        'property_area' : np.random.choice(['urban','semi-urban','rural'],n,p=[0.4,0.35,0.25]),
        'has_co_applicant': np.random.choice([0,1],n,p=[0.6,0.4]),
    }

    df = pd.DataFrame(data)

    # Derived Features
    df['debt_to_income']=(df['monthly_emi_existing']*12)/(df['annual_income_lakhs']*100000)
    df['loan_to_income'] = df['loan_amount_lakhs']/df['annual_income_lakhs']

    # Target: loan default (~15% default rate)
    default_score = (
        (df['credit_score']<600).astype(int) * 3 + 
        (df['debt_to_income']>0.5).astype(int) * 2 +
        (df['loan_to_income']>5).astype(int) * 2 +
        (df['existing_loans']>=3).astype(int) * 1.5 +
        (df['credit_history_years']<2).astype(int) * 1 +
        (df['age']<25).astype(int) * 0.5 +
        np.random.normal(0,2,n)
    )
    df['default'] = (default_score > 4.5).astype(int)

    # Missing value (~3.5%)
    for col in ['annual_income_lakhs','credit_score','credit_history_years']:
        mask = np.random.random(n) < 0.035
        df.loc[mask,col] = np.nan

    return df
    

In [9]:
def save_data(df,output_dir='data/raw'):
    os.makedirs(output_dir,exist_ok=True)
    filepath=os.path.join(output_dir,'loan_applications.csv')
    df.to_csv(filepath,index=False)
    print(f'Saved : {filepath} ({len(df)} records) , {df['default'].mean():.1%} default rate')
    return filepath

In [10]:
df=generate_loan_data()

Generating 227000 synthetic loan dataset.....


In [11]:
save_data(df)

Saved : data/raw\loan_applications.csv (227000 records) , 16.4% default rate


'data/raw\\loan_applications.csv'